In [44]:
import mne
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import pandas as pd
import os
import re

import torch
import torch.nn as nn
from scipy import signal
from scipy.stats import skew, kurtosis
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, train_test_split, KFold, GroupKFold
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score,  balanced_accuracy_score, f1_score
import numpy as np
from itertools import combinations
from scipy.signal import butter, filtfilt


In [45]:
%cd /gpfs/scratch/np3106/automatic_sleep_scoring

/gpfs/scratch/np3106/automatic_sleep_scoring


In [46]:
import os
print(os.getcwd())

/gpfs/scratch/np3106/automatic_sleep_scoring


In [47]:
import random
import numpy as np
import torch

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    
    # For full determinism (slower but stable)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

## Load data and manual labels

In [48]:
# Load manual sleep stage classification csv
CSV_PATH = Path("eeg_epochs_dataset.csv")
DATA_ROOT = Path("current_study_data_raw/") 
df = pd.read_csv(CSV_PATH)
# Sleep stage mapping: W=0, any NREM (N1/N2/N3)=1
LABEL_MAP = {"W": 0, "N1": 1, "N2": 1, "N3": 1}

df = df[df["stage_label"].isin(LABEL_MAP.keys())].copy()
df["y"] = df["stage_label"].map(LABEL_MAP).astype(int)

df["epoch_start_time"] = pd.to_numeric(df["epoch_start_time"], errors="coerce")
df["epoch_end_time"]   = pd.to_numeric(df["epoch_end_time"], errors="coerce")
df["y"]                = pd.to_numeric(df["y"], errors="coerce")

df = df.dropna(subset=["epoch_start_time", "epoch_end_time", "y"])

df["subject_id"] = pd.to_numeric(df["subject_id"], errors="coerce")
df = df.dropna(subset=["subject_id"])
df["subject_id"] = df["subject_id"].astype(int)

In [49]:
subject_list = []
for folder in os.listdir(DATA_ROOT):
    m = re.search(r"\d+", folder)
    if m:
        subject_list.append(int(m.group()))
subject_set = set(subject_list)

In [50]:
import warnings

warnings.filterwarnings(
    "ignore",
    message="No coordinate information found for channels*",
    category=RuntimeWarning,
)
warnings.filterwarnings(
    "ignore",
    message="Not setting positions of*misc channels found in montage*",
    category=RuntimeWarning,
)

warnings.filterwarnings(
    "ignore",
    message="Not setting positions of 4 misc channels found in montage*",
    category=RuntimeWarning,
)

warnings.filterwarnings(
    "ignore",
    message="Not setting positions of 6 misc channels found in montage*",
    category=RuntimeWarning,
)

## Create datasets

In [54]:
def load_raw_brainvision(subject_id: int) -> mne.io.BaseRaw:
    subdir = DATA_ROOT / f"H{subject_id:03d}"
    candidates = [
        f"{subject_id:03d}_scan.vhdr",
        f"H{subject_id:03d}_scan.vhdr",
        f"sub-{subject_id:03d}.vhdr",
        "scanner.vhdr",
        "scan.vhdr",
    ]

    for fname in candidates:
        vhdr_path = subdir / fname
        if vhdr_path.exists():
            return mne.io.read_raw_brainvision(
                vhdr_path, preload=False, verbose=False
            )
    raise FileNotFoundError(f"No .vhdr found for subject {subject_id} in {subdir}")

X_list, y_list, subject_id_list = [], [], []

# 16 channels at 200 Hz — shared across RF, CNN, and BIOT for a fair comparison.
# Matches the braindecode/biot-pretrained-prest-16chs pretrained config exactly.
TARGET_CHANNELS = [
    "F3", "F4", "C3", "C4", "P3", "P4", "O1", "O2",   # 8 EEG
    "Fz", "Cz", "Pz",                                   # 3 midline EEG
    "EOG1", "EOG2",                                      # 2 EOG
    "EMG1", "EMG2", "EMG3",                              # 3 EMG
]
SFREQ = 200
if not os.path.exists("X_arr_w_vs_nrem.npy"):
    for subject_id, sdf in df.groupby("subject_id"):
        subject_id = int(subject_id)
        if subject_id not in subject_set:
            continue
        print("Loading subject", subject_id)
        raw = load_raw_brainvision(subject_id)

        missing = [ch for ch in TARGET_CHANNELS if ch not in raw.ch_names]
        if missing:
            print(f"  Skip subject {subject_id}: missing channels {missing}")
            continue

        raw.pick(TARGET_CHANNELS)
        raw.resample(SFREQ)

        sfreq = raw.info["sfreq"]
        n_samp = int(round(30.0 * sfreq))
        n_total = raw.n_times

        for row in sdf.itertuples(index=False):
            start_samp = int(float(row.epoch_start_time) * sfreq)
            if start_samp < 0 or start_samp + n_samp > n_total:
                continue

            seg = raw.get_data(start=start_samp, stop=start_samp + n_samp)
            X_list.append(seg)
            y_list.append(int(row.y))
            subject_id_list.append(subject_id)
    np.save("X_arr_w_vs_nrem.npy", np.array(X_list, dtype=np.float32))
    np.save("y_arr_w_vs_nrem.npy", np.array(y_list, dtype=np.int64))
    np.save("subject_id_arr_w_vs_nrem.npy", np.array(subject_id_list, dtype=np.int64))
    print("epochs:", len(X_list), "labels:", len(y_list))
    print("example seg shape:", X_list[0].shape)
    print("label counts:", np.bincount(np.array(y_list)))
    X = np.array(X_list)
    y=np.array(y_list)
else:
    print("Loading preprocessed data from .npy files")
    X = np.load("X_arr_w_vs_nrem.npy")
    y = np.load("y_arr_w_vs_nrem.npy")
    subject_id_list = np.load("subject_id_arr_w_vs_nrem.npy").tolist()

Loading preprocessed data from .npy files


In [55]:
X.shape

(644, 16, 6000)

In [56]:
y.shape

(644,)

In [57]:
# 1. Prepare the 'groups' array (must match the length of X and y)
# You'll need to create this during your data loading loop
# For example: groups = np.array(subject_id_list)
groups = np.array(subject_id_list) # Assuming you tracked these in X_list loop
kf = GroupKFold(n_splits=5)

# One-time: compute subject-wise folds once and reuse everywhere
splits = list(kf.split(np.zeros(len(y)), y, groups=groups))

# Optional sanity check: print test subjects per fold
for f, (_, va) in enumerate(splits):
    print(f"Fold {f+1} test subjects:", np.unique(groups[va]))

Fold 1 test subjects: [ 17  24  34  43  51  65 109 116]
Fold 2 test subjects: [ 12  22  23  32  49  63  97 117 120]
Fold 3 test subjects: [ 16  28  38  46  61  91 100 112 119]
Fold 4 test subjects: [ 19  26  47  48  57  69  93 107]
Fold 5 test subjects: [  6  36  42  53  55  67  95 115]


## Random Forest approach

In [58]:
def extract_eeg_features(epoch, fs=200):
    """
    Extract time-domain and frequency-domain features from a single epoch
    
    Args:
        epoch: shape (n_channels, n_samples)
        fs: sampling frequency in Hz
    
    Returns:
        features: 1D array of extracted features
    """
    features = []
    theta_sum = 0.0
    alpha_sum = 0.0
    
    # Process each channel
    for channel in range(epoch.shape[0]):
        signal_data = epoch[channel, :]
        
        # === TIME DOMAIN FEATURES ===
        # Statistical features
        features.append(np.mean(signal_data))
        features.append(np.std(signal_data))
        features.append(np.var(signal_data))
        features.append(skew(signal_data))
        features.append(kurtosis(signal_data))
        features.append(np.max(signal_data))
        features.append(np.min(signal_data))
        features.append(np.ptp(signal_data))  # peak-to-peak
        features.append(np.median(signal_data))
        
        # Root mean square
        features.append(np.sqrt(np.mean(signal_data**2)))
        
        # Zero crossing rate
        zero_crossings = np.sum(np.diff(np.sign(signal_data)) != 0)
        features.append(zero_crossings)
        
        # === FREQUENCY DOMAIN FEATURES ===
        # Compute power spectral density
        freqs, psd = signal.welch(signal_data, fs=fs, nperseg=min(256, len(signal_data)))
        
        # Define frequency bands (Hz)
        # Adjust these based on your sampling rate and sleep stage research
        delta_band = (0.5, 4)    # Deep sleep
        theta_band = (4, 8)      # Light sleep, drowsiness
        alpha_band = (8, 13)     # Relaxed wakefulness
        beta_band = (13, 30)     # Active thinking, alert
        gamma_band = (30, 50)    # High-level information processing
        
        bands = [delta_band, theta_band, alpha_band, beta_band, gamma_band]
        
        for low, high in bands:
            # Find indices for this frequency band
            idx = np.logical_and(freqs >= low, freqs <= high)
            
            # Band power (absolute)
            band_power = np.trapezoid(psd[idx], freqs[idx])
            features.append(band_power)
        
        # Total power
        total_power = np.trapezoid(psd, freqs)
        features.append(total_power)
        
        # Relative band powers (percentage of total power)
        for low, high in bands:
            idx = np.logical_and(freqs >= low, freqs <= high)
            band_power = np.trapezoid(psd[idx], freqs[idx])
            relative_power = band_power / (total_power + 1e-10)  # avoid division by zero
            features.append(relative_power)
        
    return np.array(features)


# Extract features for all epochs
print("Extracting features from all epochs...")
X_features = []

for i in range(X.shape[0]):
    epoch_features = extract_eeg_features(X[i], fs=SFREQ)
    X_features.append(epoch_features)
    
    if (i + 1) % 100 == 0:
        print(f"Processed {i + 1}/{X.shape[0]} epochs")

X_features = np.array(X_features)
print(f"Feature matrix shape: {X_features.shape}")

fold_acc = []
fold_bacc = []
fold_f1 = []
rf_all_true, rf_all_pred = [], []

for fold, (train_idx, val_idx) in enumerate(splits):
    print(f"\n--- RF Fold {fold+1}/5 (Test subjects: {np.unique(groups[val_idx])}) ---")

    X_tr, X_va = X_features[train_idx], X_features[val_idx]
    y_tr, y_va = y[train_idx], y[val_idx]

    rf = RandomForestClassifier(
        n_estimators=200,
        max_depth=15,
        min_samples_split=5,
        min_samples_leaf=2,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    )

    rf.fit(X_tr, y_tr)
    y_pred = rf.predict(X_va)

    rf_all_true.extend(y_va)
    rf_all_pred.extend(y_pred)

    acc = accuracy_score(y_va, y_pred)
    bacc = balanced_accuracy_score(y_va, y_pred)
    f1 = f1_score(y_va, y_pred)   # for binary (pos label=1)

    fold_acc.append(acc)
    fold_bacc.append(bacc)
    fold_f1.append(f1)

    print(f"Accuracy: {acc:.4f}")
    print(classification_report(y_va, y_pred, target_names=['Class 0', 'Class 1']))
    print(confusion_matrix(y_va, y_pred))

print("\nRF Subject-wise CV:")
print(f"Accuracy:       {np.mean(fold_acc):.4f} (+/- {np.std(fold_acc):.4f})")
print(f"Balanced Acc:   {np.mean(fold_bacc):.4f} (+/- {np.std(fold_bacc):.4f})")
print(f"F1 (class=1):   {np.mean(fold_f1):.4f} (+/- {np.std(fold_f1):.4f})")

Extracting features from all epochs...
Processed 100/644 epochs
Processed 200/644 epochs
Processed 300/644 epochs
Processed 400/644 epochs
Processed 500/644 epochs
Processed 600/644 epochs
Feature matrix shape: (644, 352)

--- RF Fold 1/5 (Test subjects: [ 17  24  34  43  51  65 109 116]) ---
Accuracy: 0.7812
              precision    recall  f1-score   support

     Class 0       0.83      0.92      0.87       105
     Class 1       0.27      0.13      0.18        23

    accuracy                           0.78       128
   macro avg       0.55      0.53      0.53       128
weighted avg       0.73      0.78      0.75       128

[[97  8]
 [20  3]]

--- RF Fold 2/5 (Test subjects: [ 12  22  23  32  49  63  97 117 120]) ---
Accuracy: 0.6103
              precision    recall  f1-score   support

     Class 0       0.61      0.95      0.75        82
     Class 1       0.56      0.09      0.16        54

    accuracy                           0.61       136
   macro avg       0.58      0.5

In [59]:
from sklearn.metrics import cohen_kappa_score
print(f"RF Cohen's Kappa (pooled across all folds): {cohen_kappa_score(rf_all_true, rf_all_pred):.4f}")

RF Cohen's Kappa (pooled across all folds): 0.1505


## CNN approach

In [60]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)

input_channels = X.shape[1]  

# Normalization
X_norm = (X - X.mean(axis=2, keepdims=True)) / (X.std(axis=2, keepdims=True) + 1e-8)

fold_accuracies = []
fold_bacc = []
fold_f1 = []
cnn_all_true, cnn_all_pred = [], []

epochs = 100 
batch_size = 64

# Convert data to tensors
X_tensor = torch.FloatTensor(X_norm).to(device) 
y_tensor = torch.LongTensor(y).to(device)

for fold, (train_idx, val_idx) in enumerate(splits):
    seed = 42 + fold  # different but deterministic per fold
    set_seed(seed)
    print(f"\n--- Starting Fold {fold + 1}/5 (Testing on Subject IDs: {np.unique(groups[val_idx])}) ---")
    
    # Split data
    train_X, val_X = X_tensor[train_idx], X_tensor[val_idx]
    train_y, val_y = y_tensor[train_idx], y_tensor[val_idx]
    
    # Re-initialize Model for a fresh start
    model = nn.Sequential(
        nn.Conv1d(input_channels, 32, kernel_size=50, stride=6),
        nn.ReLU(),
        nn.MaxPool1d(8),
        nn.Conv1d(32, 64, kernel_size=8),
        nn.ReLU(),
        nn.MaxPool1d(8),
        nn.Flatten(),
        nn.LazyLinear(128),
        nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(128, 2)
    ).to(device)
    
    counts = torch.bincount(train_y)
    weights = (counts.sum() / (2.0 * counts.float())).to(device)
    criterion = nn.CrossEntropyLoss(weight=weights)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)
    
    # Training
    model.train()
    for epoch in range(epochs):
        perm = torch.randperm(len(train_X), device=train_X.device)
        for i in range(0, len(train_X), batch_size):
            idx = perm[i:i+batch_size]
            batch_X = train_X[idx]
            batch_y = train_y[idx]
            
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            
    # Evaluation
    model.eval()
    with torch.no_grad():
        val_outputs = model(val_X)
        val_preds = torch.argmax(val_outputs, dim=1)
        
        val_y_np = val_y.cpu().numpy()
        val_preds_np = val_preds.cpu().numpy()

        cnn_all_true.extend(val_y_np)
        cnn_all_pred.extend(val_preds_np)

        accuracy = accuracy_score(val_y_np, val_preds_np)
        bacc = balanced_accuracy_score(val_y_np, val_preds_np)
        f1 = f1_score(val_y_np, val_preds_np)
        
        fold_accuracies.append(accuracy)
        fold_bacc.append(bacc)
        fold_f1.append(f1)
        
        print(f"Fold {fold + 1} Subject-Wise Accuracy: {accuracy:.4f} | Balanced Acc: {bacc:.4f} | F1: {f1:.4f}")
        print("\nClassification Report:")
        print(classification_report(val_y_np, val_preds_np, target_names=['Class 0', 'Class 1']))
        print("\nConfusion Matrix:")
        print(confusion_matrix(val_y_np, val_preds_np))

print(f"\nFinal Subject-Wise CV Accuracy: {np.mean(fold_accuracies):.4f} (+/- {np.std(fold_accuracies):.4f})")
print(f"Balanced Accuracy: {np.mean(fold_bacc):.4f} (+/- {np.std(fold_bacc):.4f})")
print(f"F1 Score: {np.mean(fold_f1):.4f} (+/- {np.std(fold_f1):.4f})")

Using device: cuda

--- Starting Fold 1/5 (Testing on Subject IDs: [ 17  24  34  43  51  65 109 116]) ---
Fold 1 Subject-Wise Accuracy: 0.6406 | Balanced Acc: 0.5263 | F1: 0.2581

Classification Report:
              precision    recall  f1-score   support

     Class 0       0.83      0.70      0.76       105
     Class 1       0.21      0.35      0.26        23

    accuracy                           0.64       128
   macro avg       0.52      0.53      0.51       128
weighted avg       0.72      0.64      0.67       128


Confusion Matrix:
[[74 31]
 [15  8]]

--- Starting Fold 2/5 (Testing on Subject IDs: [ 12  22  23  32  49  63  97 117 120]) ---
Fold 2 Subject-Wise Accuracy: 0.6250 | Balanced Acc: 0.6131 | F1: 0.5405

Classification Report:
              precision    recall  f1-score   support

     Class 0       0.70      0.67      0.68        82
     Class 1       0.53      0.56      0.54        54

    accuracy                           0.62       136
   macro avg       0.61   

In [61]:
from sklearn.metrics import cohen_kappa_score
print(f"CNN Cohen's Kappa (pooled across all folds): {cohen_kappa_score(cnn_all_true, cnn_all_pred):.4f}")

CNN Cohen's Kappa (pooled across all folds): 0.1108


## Transfer Learning with BIOT (pretrained on EEG + sleep data)

**BIOT** (`braindecode/biot-pretrained-prest-16chs`) is a PyTorch foundation model pretrained on the
TUH Abnormal EEG Corpus and the Sleep Heart Health Study (SHHS, 5 M samples).
Its encoder processes multi-channel biosignals and its weights are freely downloadable from HuggingFace.

**Why not U-Sleep / SleepTransformer?**
- U-Sleep (perslev/U-Time) has **no downloadable pretrained weights** — the pretrained model is cloud-API only.
- SleepTransformer weights exist but require **TensorFlow 1.x** (incompatible with this notebook).
- BIOT is the closest match: PyTorch, pretrained on sleep EEG data, and publicly available.

**Strategy:**
1. Uses the shared `X`, `y`, `groups`, `splits` from the data loading cell (same data as RF and CNN).
2. Load BIOT with pretrained weights; replace its head for binary W vs N1 output.
3. Two-stage fine-tuning: freeze backbone → train head only, then unfreeze all → fine-tune.
4. Best-checkpoint tracking per fold; evaluate with the same subject-wise GroupKFold splits.

Install once if needed: `uv add braindecode` (or `pip install braindecode`)

In [62]:
import copy
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import balanced_accuracy_score, f1_score, classification_report

try:
    from braindecode.models import BIOT
except ImportError as e:
    raise ImportError(
        "braindecode is required. Install it with:  uv add braindecode  or  pip install braindecode"
    ) from e

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

# ---------------------------------------------------------------
# 1. Build model: BIOT backbone + new binary head
# ---------------------------------------------------------------
def build_biot_for_binary() -> nn.Module:
    """
    Load BIOT pretrained on EEG/sleep data (SHHS + TUH Abnormal EEG).
    Replace the classification head for 2-class (W vs N1) output.
    """
    model = BIOT.from_pretrained("braindecode/biot-pretrained-prest-16chs")
    model.return_feature = False          # make forward() return logits only

    # Discover the embedding size from the existing head's Linear layer
    in_features = None
    for submodule in model.final_layer.modules():
        if isinstance(submodule, nn.Linear):
            in_features = submodule.in_features
            break
    if in_features is None:
        raise RuntimeError("Could not find Linear layer in BIOT.final_layer")

    # Replace head for binary classification
    model.final_layer = nn.Sequential(nn.ELU(), nn.Linear(in_features, 2))
    return model


def freeze_backbone(model: nn.Module) -> None:
    """Freeze encoder; keep final_layer (new head) trainable."""
    for p in model.parameters():
        p.requires_grad = False
    for p in model.final_layer.parameters():
        p.requires_grad = True


def unfreeze_all(model: nn.Module) -> None:
    """Unfreeze all floating-point parameters (integer buffers cannot require grad)."""
    for p in model.parameters():
        if p.is_floating_point():
            p.requires_grad = True


def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        logits = model(xb)
        loss   = criterion(logits, yb)
        loss.backward()
        optimizer.step()


def evaluate(model, loader):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb     = xb.to(DEVICE)
            logits = model(xb)
            pred   = torch.argmax(logits, dim=1).cpu().numpy()
            y_pred.append(pred)
            y_true.append(yb.numpy())
    y_true = np.concatenate(y_true)
    y_pred = np.concatenate(y_pred)
    return (
        balanced_accuracy_score(y_true, y_pred),
        f1_score(y_true, y_pred, average="macro"),
        y_true,
        y_pred,
    )


# ---------------------------------------------------------------
# 2. GroupKFold transfer-learning loop
#    Uses the shared X, y, groups, splits from the data loading cell.
# ---------------------------------------------------------------
X_norm = (X - X.mean(axis=2, keepdims=True)) / (X.std(axis=2, keepdims=True) + 1e-8)

BATCH_SIZE  = 16    # keep small — BIOT on 16×6000 tensors is memory-heavy
EPOCHS_HEAD = 5     # Stage A: train head only (backbone frozen)
EPOCHS_FULL = 15    # Stage B: fine-tune entire model
LR_HEAD     = 1e-3
LR_FULL     = 1e-4
WEIGHT_DECAY = 1e-4

fold_metrics = []
all_true, all_pred = [], []

for fold, (tr_idx, va_idx) in enumerate(splits, start=1):
    set_seed(42 + fold)
    print(f"\n--- Fold {fold}/5  (Test subjects: {np.unique(groups[va_idx])}) ---")

    fold_model = build_biot_for_binary().to(DEVICE)

    counts = np.bincount(y[tr_idx])
    cls_w  = torch.tensor(
        counts.sum() / (len(counts) * counts), dtype=torch.float32, device=DEVICE
    )
    criterion = nn.CrossEntropyLoss(weight=cls_w)

    X_tr = torch.tensor(X_norm[tr_idx], dtype=torch.float32)
    y_tr = torch.tensor(y[tr_idx],      dtype=torch.long)
    X_va = torch.tensor(X_norm[va_idx], dtype=torch.float32)
    y_va = torch.tensor(y[va_idx],      dtype=torch.long)

    tr_loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=BATCH_SIZE, shuffle=True)
    va_loader = DataLoader(TensorDataset(X_va, y_va), batch_size=BATCH_SIZE, shuffle=False)

    best_bacc, best_state = -1.0, None

    # ── Stage A: head only ──────────────────────────────────────
    freeze_backbone(fold_model)
    opt = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, fold_model.parameters()),
        lr=LR_HEAD, weight_decay=WEIGHT_DECAY,
    )
    for _ in range(EPOCHS_HEAD):
        train_one_epoch(fold_model, tr_loader, opt, criterion)
        bacc, _, _, _ = evaluate(fold_model, va_loader)
        if bacc > best_bacc:
            best_bacc  = bacc
            best_state = copy.deepcopy(fold_model.state_dict())

    # ── Stage B: fine-tune all layers ───────────────────────────
    unfreeze_all(fold_model)
    opt = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, fold_model.parameters()),
        lr=LR_FULL, weight_decay=WEIGHT_DECAY,
    )
    for _ in range(EPOCHS_FULL):
        train_one_epoch(fold_model, tr_loader, opt, criterion)
        bacc, _, _, _ = evaluate(fold_model, va_loader)
        if bacc > best_bacc:
            best_bacc  = bacc
            best_state = copy.deepcopy(fold_model.state_dict())

    # ── Evaluate best checkpoint ─────────────────────────────────
    fold_model.load_state_dict(best_state)
    bacc, f1m, y_t, y_p = evaluate(fold_model, va_loader)
    fold_metrics.append((bacc, f1m))
    all_true.append(y_t)
    all_pred.append(y_p)
    print(f"  Fold {fold}: bAcc={bacc:.4f}  F1-macro={f1m:.4f}")

all_true = np.concatenate(all_true)
all_pred = np.concatenate(all_pred)

print("\n" + "=" * 50)
print("BIOT Transfer Learning — 5-Fold CV Summary")
print("=" * 50)
print(f"  Mean bAcc:     {np.mean([m[0] for m in fold_metrics]):.4f}  (+/- {np.std([m[0] for m in fold_metrics]):.4f})")
print(f"  Mean F1-macro: {np.mean([m[1] for m in fold_metrics]):.4f}  (+/- {np.std([m[1] for m in fold_metrics]):.4f})")
print("\nClassification Report (pooled across all folds):")
print(classification_report(all_true, all_pred, target_names=["W", "N1"], digits=4))

Device: cuda

--- Fold 1/5  (Test subjects: [ 17  24  34  43  51  65 109 116]) ---
  Fold 1: bAcc=0.5596  F1-macro=0.5484

--- Fold 2/5  (Test subjects: [ 12  22  23  32  49  63  97 117 120]) ---
  Fold 2: bAcc=0.6183  F1-macro=0.6181

--- Fold 3/5  (Test subjects: [ 16  28  38  46  61  91 100 112 119]) ---
  Fold 3: bAcc=0.5526  F1-macro=0.5523

--- Fold 4/5  (Test subjects: [ 19  26  47  48  57  69  93 107]) ---
  Fold 4: bAcc=0.6131  F1-macro=0.6139

--- Fold 5/5  (Test subjects: [  6  36  42  53  55  67  95 115]) ---
  Fold 5: bAcc=0.6959  F1-macro=0.6935

BIOT Transfer Learning — 5-Fold CV Summary
  Mean bAcc:     0.6079  (+/- 0.0515)
  Mean F1-macro: 0.6052  (+/- 0.0530)

Classification Report (pooled across all folds):
              precision    recall  f1-score   support

           W     0.7225    0.7828    0.7514       419
          N1     0.5211    0.4400    0.4771       225

    accuracy                         0.6630       644
   macro avg     0.6218    0.6114    0.6143   

In [63]:
from sklearn.metrics import cohen_kappa_score
print(f"BIOT Cohen's Kappa (pooled across all folds): {cohen_kappa_score(all_true, all_pred):.4f}")

BIOT Cohen's Kappa (pooled across all folds): 0.2311
